In [1]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import sys
sys.path.append("..") 

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [2]:
from UCI.pre_processing import  load_UCI_dataset, create_dataloaders

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 500)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [01:49<00:00, 87.87it/s] 


Skipped recordings: 4


100%|██████████| 1200/1200 [00:13<00:00, 86.52it/s]


Skipped recordings: 1


100%|██████████| 1200/1200 [00:13<00:00, 86.07it/s]


Skipped recordings: 0


In [4]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# The shape should be : (batch, channels, length), length = 8*125 = 1000samples, only one channel as PPG and 

X_train: torch.Size([522837, 1, 1000])
y_train: torch.Size([522837, 2])
X_val: torch.Size([66013, 1, 1000])
y_val: torch.Size([66013, 2])
X_test: torch.Size([66296, 1, 1000])
y_test: torch.Size([66296, 2])


In [5]:
print(X_train.dtype)
print(y_train.dtype)

print(torch.isnan(X_train).any())
print(torch.isnan(y_train).any())

torch.float32
torch.float32
tensor(False)
tensor(False)


In [6]:
cat ConvTran/Models/model.py

import numpy as np
from torch import nn
from Models.AbsolutePositionalEncoding import tAPE, AbsolutePositionalEncoding, LearnablePositionalEncoding
from Models.Attention import Attention, Attention_Rel_Scl, Attention_Rel_Vec


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class Permute(nn.Module):
    def forward(self, x):
        return x.permute(1, 0, 2)


def model_factory(config):
    if config['Net_Type'][0] == 'T':
        model = Transformer(config, output_size=config['output_size'])
    elif config['Net_Type'][0] == 'CC-T':
        model = CasualConvTran(config, output_size=config['output_size'])
    else:
        model = ConvTran(config, output_size=config['output_size'])
    return model


class Transformer(nn.Module):
    def __init__(self, config, output_size):
        super().__init__()
        # Parameters Initialization -----------------------------------------------
        channel_size, seq_len = config['Data_sh

In [7]:
train_loader, val_loader, test_loader = create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=64
)

X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)

X: torch.Size([64, 1, 1000]) torch.float32
y: torch.Size([64, 2]) torch.float32


In [8]:
config = {
    # Input
    'Data_shape': (32, 1, 1000),

    # ConvTran architecture
    'emb_size': 16,
    'num_heads': 8,
    'dim_ff': 256,

    # Positional encoding
    'Fix_pos_encode': 'tAPE',
    'Rel_pos_encode': 'eRPE',

    # Dropout
    'dropout': 0.01,

    # Regression
    'output_size': 2,

    # Model type
    'Net_Type': ['C-T'],
}

In [9]:
import sys

sys.path.insert(
    0,
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/ConvTran/ConvTran"
)
from Models.model import model_factory
model = model_factory(config)

X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input :", X_batch.shape)
print("Output:", output.shape)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4215.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1101.)
  return F.conv2d(


Input : torch.Size([64, 1, 1000])
Output: torch.Size([64, 2])


In [10]:
import torch.nn as nn

criterion = nn.SmoothL1Loss()

loss = criterion(output, y_batch)

print("Prediction shape:", output.shape)
print("Target shape:", y_batch.shape)
print("Loss:", loss.item())

loss.backward()

print("Backward pass successful")

Prediction shape: torch.Size([64, 2])
Target shape: torch.Size([64, 2])
Loss: 95.69707489013672
Backward pass successful


In [11]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.step()

print("Optimizer step successful")

Optimizer step successful


In [12]:
model.train()

X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()

pred = model(X_batch)

loss = criterion(pred, y_batch)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Training step successful")

Loss: 95.66804504394531
Training step successful


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_factory(config).to(device)

In [14]:
loss_module = torch.nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [15]:

from Training import SupervisedTrainer, validate, train_runner
trainer = SupervisedTrainer(
    model=model,
    dataloader=train_loader,
    device=device,
    loss_module=loss_module,
    optimizer=optimizer,
    l2_reg=None
)

val_evaluator = SupervisedTrainer(
    model=model,
    dataloader=val_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None
)

In [16]:
import os

config['epochs'] = 5
config['optimizer'] = optimizer
config['loss_module'] = loss_module
config['key_metric'] = 'loss'
config['save_dir'] = './checkpoints'

os.makedirs(config['save_dir'], exist_ok=True)

In [17]:
# metrics = trainer.train_epoch(1)

# print(metrics)

In [18]:
# val_metrics, results = val_evaluator.evaluate(1)

# print(val_metrics)

In [19]:
config['epochs'] = 5

train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/5 [00:00<?, ?it/s]2026-09-14 10:36:23,697 | INFO : Validation Summary: epoch: 1.000000 | loss: 61.711940 | SBP_MAE: 96.085083 | DBP_MAE: 28.338797 | SBP_RMSE: 98.648598 | DBP_RMSE: 30.383757 | 
2026-09-14 10:36:23,806 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 81.635114 | SBP_MAE: 114.758652 | DBP_MAE: 49.511585 | SBP_RMSE: 117.305046 | DBP_RMSE: 51.844036 | 
Training Epoch:  20%|██        | 1/5 [30:29<2:01:58, 1829.73s/it]


Best validation loss: 61.71194005314741
Saving best model for epoch: 1



2026-09-14 11:06:41,102 | INFO : Validation Summary: epoch: 2.000000 | loss: 17.998982 | SBP_MAE: 28.772562 | DBP_MAE: 8.210490 | SBP_RMSE: 35.289707 | DBP_RMSE: 10.946333 | 
2026-09-14 11:06:41,253 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 37.396249 | SBP_MAE: 63.567886 | DBP_MAE: 12.214581 | SBP_RMSE: 70.154144 | DBP_RMSE: 16.342009 | 
Training Epoch:  40%|████      | 2/5 [1:00:47<1:31:07, 1822.50s/it]


Best validation loss: 17.99898189821197
Saving best model for epoch: 2



2026-09-14 11:37:13,950 | INFO : Validation Summary: epoch: 3.000000 | loss: 12.133261 | SBP_MAE: 17.386831 | DBP_MAE: 7.858761 | SBP_RMSE: 22.093727 | DBP_RMSE: 11.088240 | 
2026-09-14 11:37:14,094 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 12.954453 | SBP_MAE: 18.748152 | DBP_MAE: 8.140602 | SBP_RMSE: 24.108824 | DBP_RMSE: 11.120488 | 
Training Epoch:  60%|██████    | 3/5 [1:31:20<1:00:54, 1827.22s/it]


Best validation loss: 12.133261141131053
Saving best model for epoch: 3



2026-09-14 12:08:14,333 | INFO : Validation Summary: epoch: 4.000000 | loss: 10.918485 | SBP_MAE: 15.579700 | DBP_MAE: 7.233953 | SBP_RMSE: 19.748106 | DBP_RMSE: 10.022408 | 
2026-09-14 12:08:14,479 | INFO : Epoch 4 Training Summary: epoch: 4.000000 | loss: 11.223414 | SBP_MAE: 15.782532 | DBP_MAE: 7.640980 | SBP_RMSE: 20.146709 | DBP_RMSE: 10.636860 | 
Training Epoch:  80%|████████  | 4/5 [2:02:20<30:40, 1840.32s/it]  


Best validation loss: 10.918485259977336
Saving best model for epoch: 4



2026-09-14 12:42:14,088 | INFO : Validation Summary: epoch: 5.000000 | loss: 10.635339 | SBP_MAE: 15.110643 | DBP_MAE: 7.136041 | SBP_RMSE: 19.428556 | DBP_RMSE: 9.919663 | 
2026-09-14 12:42:14,202 | INFO : Epoch 5 Training Summary: epoch: 5.000000 | loss: 10.696024 | SBP_MAE: 15.034608 | DBP_MAE: 7.332968 | SBP_RMSE: 19.336176 | DBP_RMSE: 10.322886 | 
2026-09-14 12:42:14,205 | INFO : Train Time: 2.0 hours, 36.0 minutes, 20.13138175010681 seconds




Best validation loss: 10.635338962907985
Saving best model for epoch: 5



In [22]:
checkpoint = torch.load(
    './checkpoints/model_best.pth',
    map_location=device
)

model.load_state_dict(checkpoint['state_dict'])
model.to(device)

ConvTran(
  (embed_layer): Sequential(
    (0): Conv2d(1, 64, kernel_size=(1, 8), stride=(1, 1), padding=same)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (embed_layer2): Sequential(
    (0): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1), padding=valid)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (Fix_Position): tAPE(
    (dropout): Dropout(p=0.01, inplace=False)
  )
  (attention_layer): Attention_Rel_Scl(
    (key): Linear(in_features=16, out_features=16, bias=False)
    (value): Linear(in_features=16, out_features=16, bias=False)
    (query): Linear(in_features=16, out_features=16, bias=False)
    (dropout): Dropout(p=0.01, inplace=False)
    (to_out): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (LayerNorm): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=T

In [23]:
test_evaluator = SupervisedTrainer(
    model=model,
    dataloader=test_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None,
    print_interval=10,
    console=True,
    print_conf_mat=False
)

In [24]:
test_metrics, test_results = test_evaluator.evaluate(
    epoch_num=None,
    keep_all=True
)

In [26]:
print("Test Results:")

for key, value in test_metrics.items():
    print(f"{key}: {value}")

Test Results:
epoch: None
loss: 11.0024459202367
SBP_MAE: 15.458683967590332
DBP_MAE: 7.522745609283447
SBP_RMSE: 19.76251220703125
DBP_RMSE: 10.315971374511719


In [27]:
y_true = test_results['targets']
y_pred = test_results['predictions']

print("True shape:", y_true.shape)
print("Pred shape:", y_pred.shape)

True shape: (133841, 2)
Pred shape: (133841, 2)


In [29]:
import numpy as np
sbp_true = y_true[:, 0]
sbp_pred = y_pred[:, 0]

dbp_true = y_true[:, 1]
dbp_pred = y_pred[:, 1]

sbp_error = sbp_pred - sbp_true
dbp_error = dbp_pred - dbp_true

print("\n===== TEST RESULTS =====")

print(f"SBP MAE  : {np.mean(np.abs(sbp_error)):.2f} mmHg")
print(f"SBP RMSE : {np.sqrt(np.mean(sbp_error**2)):.2f} mmHg")
print(f"SBP ME   : {np.mean(sbp_error):.2f} mmHg")
print(f"SBP STD  : {np.std(sbp_error):.2f} mmHg")

print(f"\nDBP MAE  : {np.mean(np.abs(dbp_error)):.2f} mmHg")
print(f"DBP RMSE : {np.sqrt(np.mean(dbp_error**2)):.2f} mmHg")
print(f"DBP ME   : {np.mean(dbp_error):.2f} mmHg")
print(f"DBP STD  : {np.std(dbp_error):.2f} mmHg")


===== TEST RESULTS =====
SBP MAE  : 15.46 mmHg
SBP RMSE : 19.76 mmHg
SBP ME   : -0.13 mmHg
SBP STD  : 19.76 mmHg

DBP MAE  : 7.52 mmHg
DBP RMSE : 10.32 mmHg
DBP ME   : -0.94 mmHg
DBP STD  : 10.27 mmHg


In [ ]:
config['epochs'] = 50

train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/50 [00:00<?, ?it/s]2026-09-15 11:03:03,715 | INFO : Validation Summary: epoch: 1.000000 | loss: 58.699809 | SBP_MAE: 90.635811 | DBP_MAE: 27.763807 | SBP_RMSE: 93.422333 | DBP_RMSE: 30.016075 | 



Best validation loss: 58.69980884592626
Saving best model for epoch: 1



2026-09-15 11:03:03,979 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 80.917021 | SBP_MAE: 113.177238 | DBP_MAE: 49.656796 | SBP_RMSE: 115.865929 | DBP_RMSE: 52.141373 | 
Training Epoch:   2%|▏         | 1/50 [34:07<27:52:27, 2047.92s/it]2026-09-15 11:24:00,570 | INFO : Validation Summary: epoch: 2.000000 | loss: 14.248232 | SBP_MAE: 21.026512 | DBP_MAE: 8.453051 | SBP_RMSE: 27.132923 | DBP_RMSE: 11.555298 | 
2026-09-15 11:24:00,718 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 32.437527 | SBP_MAE: 54.663975 | DBP_MAE: 11.199454 | SBP_RMSE: 62.600822 | DBP_RMSE: 15.161275 | 
Training Epoch:   4%|▍         | 2/50 [55:04<21:06:00, 1582.52s/it]


Best validation loss: 14.248232477389873
Saving best model for epoch: 2



2026-09-15 11:39:07,259 | INFO : Validation Summary: epoch: 3.000000 | loss: 11.377100 | SBP_MAE: 16.072105 | DBP_MAE: 7.659520 | SBP_RMSE: 20.399710 | DBP_RMSE: 10.514038 | 
2026-09-15 11:39:07,386 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 12.116581 | SBP_MAE: 17.361456 | DBP_MAE: 7.850342 | SBP_RMSE: 22.183569 | DBP_RMSE: 10.899641 | 
Training Epoch:   6%|▌         | 3/50 [1:10:11<16:37:53, 1273.91s/it]


Best validation loss: 11.37710043199115
Saving best model for epoch: 3

